# How much Data is given

In [41]:
from pathlib import Path
import json

# Only jsonl files are relevant. This just remains from older code
def collect_json_files(directories):
    """Collect JSONL and JSON files from all relevant source folders."""

    collected = []
    seen = set()

    for directory in directories:
        if not directory.exists():
            print(f"Skipping missing directory: {directory}")
            continue

        for pattern in ("*.jsonl", "*.json"):
            for file_path in sorted(directory.glob(pattern)):
                resolved = file_path.resolve()
                if resolved in seen:
                    continue
                seen.add(resolved)
                collected.append(file_path)

    return collected

def find_workspace_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "default_process").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root (expected a default_process folder) "
        f"above the current directory: {start}"
    )

# For now only the standard process passes are considered
WORKSPACE_ROOT = find_workspace_root()
data_dirs = [WORKSPACE_ROOT / "default_process"]
files = collect_json_files(data_dirs)

if not files:
    print("No JSON/JSONL files found in the configured data directories.")
else:
    total_lines = 0
    total_nonempty = 0
    total_invalid = 0

    print(f"Found {len(files)} file(s):\n")

    for file_path in files:
        lines = nonempty = invalid = 0

        with file_path.open("r", encoding="utf-8-sig", errors="replace") as file:
            for line in file:
                lines += 1
                if line.strip():
                    nonempty += 1
                    try:
                        json.loads(line)
                    except json.JSONDecodeError:
                        invalid += 1

        total_lines += lines
        total_nonempty += nonempty
        total_invalid += invalid

        print(
            f"{file_path.name}: "
            f"{lines:,} lines, "
            f"{nonempty:,} non-empty records, "
            f"{invalid:,} invalid JSON lines, "
            f"{file_path.stat().st_size:,} bytes"
        )

    print("\nTotals:")
    print(f"Files: {len(files):,}")
    print(f"Lines: {total_lines:,}")
    print(f"Non-empty records: {total_nonempty:,}")
    print(f"Invalid JSON lines: {total_invalid:,}")

Found 10 file(s):

Ablauf1 100626.jsonl: 9,317 lines, 9,317 non-empty records, 0 invalid JSON lines, 39,246,240 bytes
Ablauf2 100626.jsonl: 8,547 lines, 8,547 non-empty records, 0 invalid JSON lines, 36,322,625 bytes
Ablauf2.1 08072026.jsonl: 10,012 lines, 10,012 non-empty records, 0 invalid JSON lines, 48,085,982 bytes
Ablauf3 100626.jsonl: 4,227 lines, 4,227 non-empty records, 0 invalid JSON lines, 17,183,723 bytes
Ablauf4 100626.jsonl: 8,014 lines, 8,014 non-empty records, 0 invalid JSON lines, 35,971,590 bytes
Ablauf1 100626.json: 9,317 lines, 9,317 non-empty records, 0 invalid JSON lines, 39,869,930 bytes
Ablauf2 100626.json: 8,547 lines, 8,547 non-empty records, 0 invalid JSON lines, 36,919,263 bytes
Ablauf2.1 08072026.json: 10,012 lines, 10,012 non-empty records, 0 invalid JSON lines, 48,725,929 bytes
Ablauf3 100626.json: 4,227 lines, 4,227 non-empty records, 0 invalid JSON lines, 17,451,990 bytes
Ablauf4 100626.json: 8,014 lines, 8,014 non-empty records, 0 invalid JSON lines, 3

# How long does each log take?

We estimate using the timestamp of each MQTT message

In [60]:
for file in files:
    first_timestamp = None
    last_timestamp = None

    with file.open("r", encoding="utf-8-sig", errors="replace") as stream:
        for line in stream:
            if not line.strip():
                continue
            try:
                value = json.loads(line).get("timestamp")
            except (json.JSONDecodeError, AttributeError, TypeError, ValueError):
                continue

            parser = globals().get("parse_event_time")
            timestamp = parser(value) if parser is not None else None
            if timestamp is None:
                continue

            if first_timestamp is None:
                first_timestamp = timestamp
            last_timestamp = timestamp

    if first_timestamp is None:
        print(f"{file.name}: no valid timestamps found")
    else:
        duration = last_timestamp - first_timestamp
        print(
            f"{file.name}: first={first_timestamp.isoformat()}, "
            f"last={last_timestamp.isoformat()}, duration={duration}"
        )

Ablauf1 100626.jsonl: first=2026-06-10T07:44:30.322508+00:00, last=2026-06-10T08:15:31.988093+00:00, duration=0:31:01.665585
Ablauf2 100626.jsonl: first=2026-06-10T08:16:53.831638+00:00, last=2026-06-10T08:45:34.406168+00:00, duration=0:28:40.574530
Ablauf2.1 08072026.jsonl: first=2026-07-08T08:02:46.009330+00:00, last=2026-07-08T08:34:19.384013+00:00, duration=0:31:33.374683
Ablauf3 100626.jsonl: first=2026-06-10T08:55:08.301572+00:00, last=2026-06-10T09:08:36.259361+00:00, duration=0:13:27.957789
Ablauf4 100626.jsonl: first=2026-06-10T09:13:49.282000+00:00, last=2026-06-10T09:40:21.548707+00:00, duration=0:26:32.266707
Ablauf1 100626.json: first=2026-06-10T07:44:30.322508+00:00, last=2026-06-10T08:15:31.988093+00:00, duration=0:31:01.665585
Ablauf2 100626.json: first=2026-06-10T08:16:53.831638+00:00, last=2026-06-10T08:45:34.406168+00:00, duration=0:28:40.574530
Ablauf2.1 08072026.json: first=2026-07-08T08:02:46.009330+00:00, last=2026-07-08T08:34:19.384013+00:00, duration=0:31:33.37